# S6_05 — 프롬프트(Prompts): 검증된 템플릿을 슬래시 명령으로 호출하기

이번 노트북에서는 엠시피(MCP) 서버의 마지막 핵심 컴포넌트인 프롬프트를 다룬다. 도구가 동작을 수행하고 리소스가 데이터를 노출한다면, 프롬프트는 이미 검증된 고품질 지시문 템플릿을 사용자에게 제공하는 역할을 한다. 강의노트 §2.4 와 §2.5 두 절을 코드로 옮기면서, 서버 측에서 프롬프트를 정의하는 방법과 클라이언트 측에서 그것을 호출하는 방법을 모두 익힌다. 그리고 마지막에는 도구와 리소스와 프롬프트라는 세 컴포넌트가 한 사이클 안에서 협업하는 모습을 직접 시연하게 된다.

본 노트북은 두 개의 스킬자(Skilljar) 레슨에 대응한다. 첫째는 스킬자 레슨 9 번(아이디 287784, 프롬프트 정의하기 레슨)이고, 강의노트로는 §2.4 (라인 약 1166 부터 1281 까지)에 매핑된다. 둘째는 스킬자 레슨 10 번(아이디 287786, 클라이언트에서 프롬프트 사용하기 레슨)이고, 강의노트로는 §2.5 (라인 약 1285 부터 1395 까지)에 매핑된다. 두 레슨이 짝을 이루어야 정의와 활용이 한꺼번에 완성되기 때문에 한 노트북에서 함께 다룬다.

**학습 목표**:

본 노트북을 마치면 학생은 다음 다섯 가지를 할 수 있어야 한다. 첫째, 프롬프트 데코레이터를 사용하여 검증된 프롬프트 템플릿을 정의할 수 있다. 본 노트북에서는 포맷팅 프롬프트와 요약 프롬프트와 번역 프롬프트 세 가지를 차례로 만들면서 패턴을 익힌다. 둘째, 사용자 메시지와 어시스턴트 메시지 두 가지 메시지 타입을 혼합하여 적은 예시 학습 스타일의 프롬프트를 구성할 수 있다. 셋째, 클라이언트 측 메서드인 프롬프트 목록 조회와 프롬프트 가져오기 메서드를 구현하여, 클라이언트가 서버의 프롬프트 카탈로그를 조회하고 활용할 수 있도록 만들 수 있다. 넷째, 슬래시 명령을 파싱하여 적절한 프롬프트로 디스패치하는 로직을 작성할 수 있다. 다섯째, 도구와 리소스와 프롬프트라는 세 컴포넌트가 한 사이클 안에서 어떻게 서로 협업하는지를 한 번의 흐름으로 시연할 수 있다.

**선수 학습 사항**:

본 노트북을 시작하기 전에 직전 노트북인 리소스 노트북을 먼저 완료해야 한다. 그래야 같은 폴더에 도구 두 개와 리소스 두 개가 정의된 서버 파일이 있는 상태가 되고, 클라이언트 파일에는 리소스 읽기 메서드도 이미 추가되어 있어야 한다.

**최종 산출물 약속**:

본 노트북이 종료될 시점에는 서버 파일이 도구와 리소스와 프롬프트를 모두 포함하여, 참조 구현인 도메인 폴더의 서버 파일과 동등한 완성본이 된다. 이로써 본 주차의 핵심 학습 자산이 모두 갖춰진 상태에 이르게 된다.

> [!ref] 참조 소스: 도메인 폴더의 서버 파일 라인 67 부터 91 까지, 클라이언트 파일 라인 57 부터 65 까지


## §1. 프롬프트란 무엇인가 — 사용자가 제어하는 컴포넌트

MCP 의 세 가지 컴포넌트는 *누가 호출하는가* 라는 기준으로 깔끔하게 구분된다. 강의노트 §2.4 의 핵심 메시지를 풀어 보면 이렇다.

먼저 도구(Tools)는 클로드(Claude)가 능동적으로 호출을 결정하는 컴포넌트다. 즉 사용자의 요청을 보고 LLM 이 *"이 도구를 써야겠다"* 고 스스로 판단하는 방식이다. 그래서 LLM 제어 컴포넌트라는 표현이 어울린다. 다음으로 리소스(Resources)는 애플리케이션이 자동으로 가져오는 데이터 노출 컴포넌트다. 사용자가 `@` 기호로 문서를 멘션하면 애플리케이션이 그 의도를 받아 리소스를 페치하는 것이 대표적인 예다. 그래서 앱 제어 컴포넌트라는 표현이 어울린다.

마지막으로 프롬프트(Prompts)는 사용자가 명시적으로 선택하는 컴포넌트다. 사용자가 `/format` 처럼 슬래시 명령을 직접 입력하여 프롬프트를 호출하는 방식이다. 그래서 사용자 제어 컴포넌트라는 표현이 가장 정확하다. 이 세 가지 호출 주체의 구분이 MCP 의 컴포넌트 설계 철학을 가장 잘 보여 준다.

프롬프트의 본질은 무엇일까. 강의노트 §2.4 의 표현을 빌리면, 프롬프트는 MCP 서버 작성자가 미리 만들어 두고 검증한 고품질 지시문이다. 사용자가 즉석에서 떠올려서 작성하는 프롬프트보다 더 일관되고 더 품질 높은 결과를 만들어 낸다는 것이 핵심이다.

> [!finding] 강의노트 §2.4 의 핵심 통찰
> 강의노트는 다음과 같이 정리한다. 사용자가 직접 그 작업을 수행할 수도 있지만, MCP 서버 작성자가 신중하게 개발하고 테스트한 프롬프트를 사용하면 훨씬 일관되고 품질 높은 결과를 얻을 수 있다는 것이다. 여기에는 한 가지 가정이 깔려 있다. MCP 서버 작성자가 그 도메인의 전문가라는 가정이다. 그 도메인의 전문 지식을 프롬프트라는 형태로 외부화하여 저장해 두는 것이 바로 프롬프트 컴포넌트의 본질이다.

**프롬프트의 형태와 동작 방식**:

프롬프트는 클라이언트가 곧바로 사용할 수 있는 메시지 리스트의 형태로 반환된다. 즉 user 와 assistant 메시지의 배열이다. 또한 매개변수를 받아 본문에 보간할 수 있어서, 같은 프롬프트를 다양한 인자로 재사용할 수 있다. 가장 단순한 예는 `doc_id: str` 같은 매개변수를 받아 본문에 끼워 넣는 형태다. 그리고 프롬프트 본문 안에는 어떤 도구를 사용할지, 또는 어떤 리소스를 참조할지를 명시할 수 있다. 바로 이 점이 도구와 리소스와 프롬프트의 3자 연계가 시작되는 출발점이 된다.


In [ ]:
# Setup — Week_07.md §2.4 line ~1199 — base 메시지 타입 임포트
from pydantic import Field
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base
import os

print('FastMCP + base imported. Working dir:', os.getcwd())

## §2. 포맷팅 프롬프트 — 문서를 마크다운 형식으로 재구성하라는 지시

이번에는 강의노트 §2.4 의 핵심 예제인 포맷팅 프롬프트를 구현해 보자. 사용자의 워크플로는 이렇다. 사용자가 슬래시 명령으로 포맷팅 명령을 입력하면, 이 프롬프트가 클로드(Claude)에게 전달되고, 클로드는 그 지시에 따라 문서 편집 도구를 호출하여 문서를 마크다운 형식으로 재포맷한다.

이 프롬프트의 핵심 설계 요소는 다음과 같다. 우선 매개변수로 문서 아이디를 받는다. 이 매개변수의 값은 함수 본문에서 형식 문자열 보간을 통해 프롬프트 본문에 끼워 넣어진다. 함수의 반환 타입은 메시지 리스트인데, 본 예제에서는 그 리스트 안에 사용자 메시지 하나만 들어 있다. 그리고 가장 흥미로운 부분은 프롬프트 본문에서 문서 편집 도구를 명시적으로 언급한다는 점이다. 즉 이 도구를 사용해서 문서를 편집하라고 지시하는 셈이다. 이것이 바로 프롬프트와 도구와 리소스의 3자 연계가 시작되는 결정적 지점이다.

본 셀에서 사용할 핵심 식별자를 정리해 두자. 데코레이터에는 두 가지 인자가 들어간다. 첫째는 프롬프트의 이름이고, 둘째는 사용자에게 보여지는 설명 문자열이다. 함수의 이름은 문서를 포맷한다는 의미를 담아 짓고, 반환은 사용자 메시지 하나를 담은 리스트 형태다. 참조 구현은 도메인 폴더 안의 서버 파일 라인 68 부터 91 까지에 있다.


In [ ]:
# Week_07.md §2.4 line ~1206 — format 프롬프트 (cli_project/mcp_server.py:68-91)
# 단독 데모용 FastMCP 인스턴스

demo_mcp = FastMCP('DocumentMCP-Demo', log_level='ERROR')

@demo_mcp.prompt(
    name='format',
    description='Rewrite a document in markdown format',
)
def format_document(
    doc_id: str = Field(description='ID of the document to format'),
) -> list[base.Message]:
    prompt = f'''
    Your goal is to reformat the following document in markdown format.

    The id of the document you need to format is:
    <document_id>
    {doc_id}
    </document_id>

    Add in headers, bullet points, tables, etc as necessary.
    Feel free to add in examples to clarify the content.
    Use the 'edit_document' tool to edit the document. Do not leave anything out.
    The entire contents of the document should be included in the reformatted version.

    After the document has been edited, return the full contents of the reformatted document
    as the final answer to this prompt.
    '''
    return [base.UserMessage(prompt)]

# Direct invocation (in real flow, MCP server invokes via get_prompt)
messages = format_document('plan.md')
print(f'Returned {len(messages)} message(s)')
print('First message role:', messages[0].role)
print('First message content[:200]:', str(messages[0].content)[:200])

## §3. 적은 예시 학습 프롬프트 — 요약 프롬프트와 어시스턴트 메시지의 활용

강의노트 §2.4 (라인 약 1279)에서는 다음과 같은 약속을 한다. 사용자 메시지만 사용하는 것이 아니라 어시스턴트 메시지도 함께 섞어서 적은 예시 학습 스타일의 프롬프트를 만들어 보라는 것이다. 본 셀에서는 그 약속을 이행해 본다.

적은 예시 학습 프롬프트의 핵심 패턴은 다음과 같이 세 단계로 이루어진다. 첫 번째 단계에서 사용자 메시지를 통해 예시 입력을 보여 준다. 두 번째 단계에서 어시스턴트 메시지를 통해 모델이 어떻게 답해야 하는지 그 전형을 직접 보여 준다. 바로 이 단계가 가장 중요한데, 클로드에게 이런 식으로 답해 달라는 모범 답안의 형식을 명시적으로 알려 주는 역할을 한다. 세 번째 단계에서 마지막으로 또 다른 사용자 메시지를 통해 진짜 사용자 요청을 전달한다.

이 패턴은 사실 새로운 개념이 아니다. 우리가 이미 3 주차의 프롬프트 엔지니어링 트랙에서 다룬 맥락 내 학습 기법을 엠시피 프롬프트라는 형태로 외부화한 것이다. 강의 진행이 진행될수록 이전 주차의 개념이 새로운 형태로 다시 등장하는 모습을 종종 볼 수 있는데, 이번이 그 한 사례라고 할 수 있다.


In [ ]:
# Week_07.md §2.4 line ~1279 — summarize 프롬프트 (few-shot)

@demo_mcp.prompt(
    name='summarize',
    description='Summarize a document in 3 bullet points',
)
def summarize_doc(
    doc_id: str = Field(description='ID of the document to summarize'),
) -> list[base.Message]:
    return [
        # Few-shot example: input -> desired output format
        base.UserMessage('Summarize spec.txt in exactly 3 bullets.'),
        base.AssistantMessage(
            '- Defines technical requirements for equipment\n'
            '- Lists material specifications\n'
            '- Outlines testing protocols'
        ),
        # Real user request
        base.UserMessage(f'Now summarize {doc_id} in exactly 3 bullets.'),
    ]

messages = summarize_doc('plan.md')
for i, m in enumerate(messages):
    print(f'[{i}] {m.role}: {str(m.content)[:80]}')

## §4. 번역 프롬프트 — 단순한 매개변수 보간 사례

강의노트 §2.4 의 약속 중에 다음과 같은 부분이 있었다. 동일 서버에 요약 프롬프트와 번역 프롬프트 같은 두 개의 프롬프트를 더 붙여서 프롬프트 카탈로그를 구성해 보라는 부분이다. 우리는 이미 요약 프롬프트를 §3 에서 만들어 보았으니, 이번에는 그 약속의 마지막 한 조각인 번역 프롬프트를 추가해 보자.

이 프롬프트의 구조는 매우 단순하다. 사용자 메시지 하나만 사용하며, 적은 예시 학습 같은 별도 예시도 없이 매개변수만 본문에 보간하는 형태다. 다만 매개변수가 두 개 들어간다는 점이 앞선 두 프롬프트와 다른 점이다. 첫 번째 매개변수는 번역할 문서의 아이디이고, 두 번째 매개변수는 목표 언어를 받는다. 예를 들어 첫 매개변수가 어떤 문서이고 둘째 매개변수가 한국어라면, 본문에서 두 값이 모두 보간되어 그 문서를 한국어로 번역하라는 지시가 만들어진다.

이렇게 매개변수 보간만 사용하는 단순한 프롬프트도 충분히 유용하다. 도메인의 잘 다듬어진 지시문을 한 번 작성해 두면, 사용자는 매개변수만 바꿔서 그 지시문을 반복적으로 재사용할 수 있게 되기 때문이다. 즉 한 번의 작성 노력이 무한한 재사용으로 이어진다는 의미다.


In [ ]:
# Week_07.md §2.4 line ~1279 — translate 프롬프트 (단순 UserMessage)

@demo_mcp.prompt(
    name='translate',
    description='Translate a document to a target language',
)
def translate_doc(
    doc_id: str = Field(description='ID of the document to translate'),
    target_lang: str = Field(description='Target language (e.g., Korean, French)'),
) -> list[base.Message]:
    return [
        base.UserMessage(
            f'Translate the document with id <doc_id>{doc_id}</doc_id> '
            f'into {target_lang}. Use read_doc_contents to fetch the source first, '
            f'then return only the translated text.'
        ),
    ]

messages = translate_doc('plan.md', 'Korean')
print(messages[0].role, ':', messages[0].content)

## §5. 서버 파일의 최종 통합본 — 도구와 리소스와 프롬프트를 한 파일에

앞선 노트북에서는 도구 두 개와 리소스 두 개를 포함한 서버 파일을 저장해 두었다. 이번 단계에서는 그 파일에 §2 에서 정의한 포맷팅 프롬프트를 추가하여 최종 완성본을 만든다. 이 파일은 참조 구현인 도메인 폴더의 서버 파일(라인 1 부터 91 까지)과 동일한 구조를 가지게 된다.

최종 파일의 구성을 정리하면 다음과 같다. 먼저 도구는 두 개로, 문서 읽기 도구와 문서 편집 도구가 들어 있다. 리소스도 두 개로, 직접 유알아이 형태의 문서 목록 리소스와 매개변수 유알아이 형태의 단일 문서 리소스가 그것이다. 마지막으로 프롬프트는 한 개만 추가되며, 그 이름은 포맷팅이다. 이로써 엠시피의 세 가지 핵심 컴포넌트가 한 서버 안에 모두 모이게 된다.

> [!tip] 요약 프롬프트와 번역 프롬프트는 본 노트북에만 남긴다
> 위에서 함께 만들어 본 요약 프롬프트와 번역 프롬프트 두 가지는 학습용 데모로만 본 노트북에 남겨 두고, 실제 서버 파일에는 저장하지 않는다. 그 이유는 참조 구현인 도메인 폴더의 서버 파일이 포맷팅 프롬프트만 포함하고 있기 때문이다. 두 프롬프트를 자기 손으로 추가하는 작업은 다음 노트북의 학생 자율 실습에서 자연스럽게 이어지게 된다. 즉 본 노트북에서 데모로 본 패턴을 학생이 직접 응용해 볼 기회가 곧바로 주어지는 셈이다.


In [ ]:
# Week_07.md §2.4 line ~1206 — mcp_server.py 최종본 (cli_project/mcp_server.py 라인 1-91 동등)

server_code = '''from pydantic import Field
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

mcp = FastMCP("DocumentMCP", log_level="ERROR")


docs = {
    "deposition.md":   "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf":      "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project\'s budget and expenditures.",
    "outlook.pdf":     "This document presents the projected future performance of the system.",
    "plan.md":         "The plan outlines the steps for the project\'s implementation.",
    "spec.txt":        "These specifications define the technical requirements for the equipment.",
}


# Tool: Read a doc
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string",
)
def read_document(
    doc_id: str = Field(description="ID of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]


# Tool: Edit a doc
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the document content with a new string",
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including white space"),
    new_str: str = Field(description="The text to insert in place of the old text"),
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)


# Resource: Direct — return all doc IDs
@mcp.resource(
    "docs://documents",
    mime_type="application/json",
)
def list_docs() -> list[str]:
    return list(docs.keys())


# Resource: Templated — return contents of a particular doc
@mcp.resource(
    "docs://documents/{doc_id}",
    mime_type="text/plain",
)
def fetch(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]


# Prompt: Reformat a doc as markdown
@mcp.prompt(
    name="format",
    description="Rewrite a document in markdown format",
)
def format_document(
    doc_id: str = Field(description="ID of the document to format")
) -> list[base.Message]:
    prompt = f"""
    Your goal is to reformat the following document in markdown format.

    The id of the document you need to format is:
    <document_id>
    {doc_id}
    </document_id>

    Add in headers, bullet points, tables, etc as necessary.
    Feel free to add in examples to clarify the content.
    Use the 'edit_document' tool to edit the document. Do not leave anything out.
    The entire contents of the document should be included in the reformatted version.

    After the document has been edited, return the full contents of the reformatted document
    as the final answer to this prompt.
    """
    return [base.UserMessage(prompt)]


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open('mcp_server.py', 'w', encoding='utf-8') as f:
    f.write(server_code)

print('mcp_server.py finalized: 2 tools + 2 resources + 1 prompt (format).')

## §6. 클라이언트에서 프롬프트 사용하기 — 두 개의 클라이언트 메서드

강의노트 §2.5 의 약속을 이행할 차례다. 클라이언트가 서버의 프롬프트 카탈로그를 활용하려면 두 개의 메서드가 필요하다.

첫 번째는 프롬프트 목록 조회 메서드다. 참조 구현은 도메인 폴더의 클라이언트 파일 라인 57 부터 60 까지에 있다. 이 메서드의 역할은 단순하다. 세션의 프롬프트 목록 조회 메서드를 호출하여 결과를 받고, 그 결과의 프롬프트 배열을 반환하는 것이 전부다. 이 메서드를 통해 클라이언트는 이 서버에는 어떤 프롬프트들이 정의되어 있는지를 한눈에 조회할 수 있다.

두 번째는 프롬프트 가져오기 메서드다. 참조 구현은 도메인 폴더의 클라이언트 파일 라인 62 부터 65 까지에 있다. 이 메서드는 인자가 보간된 특정 프롬프트를 가져오는 역할을 한다. 즉 프롬프트의 이름과 인자 사전을 함께 전달하면, 서버가 그 인자를 키워드 인자로 프롬프트 함수에 전달하여 보간된 메시지 리스트를 반환해 준다. 이 메서드는 결과의 메시지 필드를 반환하므로, 호출자는 그것을 바로 클로드에게 보낼 수 있는 형태가 된다.

한 가지 다행스러운 점이 있다. 앞선 노트북에서 저장해 둔 클라이언트 파일에는 이미 두 메서드가 정의되어 있다는 사실이다. 그래서 본 셀에서는 새로 메서드를 작성하는 것이 아니라, 그 구현이 실제로 잘 작동하는지를 호출 시연을 통해 확인하는 데 집중한다.


In [ ]:
# Week_07.md §2.5 line ~1297 — list_prompts / get_prompt 호출 시연
import sys
if 'mcp_client' in sys.modules:
    del sys.modules['mcp_client']
from mcp_client import MCPClient

import os
USE_UV = os.getenv('USE_UV', '0') == '1'
command, args = ('uv', ['run', 'mcp_server.py']) if USE_UV else ('python', ['mcp_server.py'])

async def demo_prompts():
    async with MCPClient(command=command, args=args) as client:
        # 1) Discover prompts
        prompts = await client.list_prompts()
        print(f'--- Found {len(prompts)} prompt(s) ---')
        for p in prompts:
            arg_names = [a.name for a in (p.arguments or [])]
            print(f'  /{p.name}  args={arg_names}  desc={p.description}')

        # 2) Fetch a specific prompt with arguments
        print('\n--- get_prompt("format", {"doc_id": "plan.md"}) ---')
        messages = await client.get_prompt('format', {'doc_id': 'plan.md'})
        for i, m in enumerate(messages):
            content_preview = str(m.content)[:200] if isinstance(m.content, str) else str(m.content)[:200]
            print(f'  [{i}] {m.role}: {content_preview}')
        return prompts, messages

prompts, messages = await demo_prompts()

## §7. 슬래시 명령 시뮬레이션 — 슬래시 입력의 파싱과 디스패치

강의노트 §2.5 (라인 약 1329 부터 1332 까지)는 다음과 같이 약속한다. 사용자가 슬래시를 입력하면 사용 가능한 프롬프트들이 명령어 형태로 나타난다는 것이다. 마치 슬랙(Slack)이나 디스코드(Discord)의 슬래시 명령처럼 동작하는 셈이다. 본 셀에서는 그 흐름을 코드로 재현해 본다.

슬래시 명령 파서의 동작 규칙을 정리하면 다음과 같다. 참조 구현은 도메인 폴더의 명령행 채팅 모듈에 있는 슬래시 처리 로직이다.

첫 번째 규칙은 입력의 시작 부분을 점검하는 것이다. 입력이 슬래시로 시작하면 슬래시 명령으로 인식한다. 두 번째 규칙은 토큰 분리다. 입력을 공백으로 분리해서 첫 번째 토큰을 프롬프트의 이름으로 사용하고, 나머지 토큰들을 위치 인자로 처리한다. 세 번째 규칙은 인자 매핑이다. 서버에서 프롬프트 목록 조회로 가져온 프롬프트 메타데이터에는 각 프롬프트의 인자 정보가 포함되어 있다. 그 메타데이터를 사용하여 위치 인자를 키워드 인자 이름으로 매핑한다. 마지막 단계로 프롬프트 가져오기 메서드를 호출하여 보간된 메시지 리스트를 받고, 그것을 클로드에게 투입한다.

구체적인 예시로 풀어 보자. 사용자가 포맷팅 명령으로 어떤 문서를 지정하여 입력했다고 하자. 그러면 첫 토큰인 포맷팅이 프롬프트 이름으로 잡히고, 두 번째 토큰인 문서 이름이 위치 인자로 잡힌다. 서버에서 가져온 포맷팅 프롬프트의 첫 번째 인자 이름이 문서 아이디라는 사실을 알고 있으므로, 결과적으로 그 문서 아이디 인자에 그 문서 이름이 매핑된 호출이 만들어진다.


In [ ]:
# Week_07.md §2.5 line ~1370 — / 슬래시 명령 파서·디스패처

async def handle_slash(user_input: str):
    """Parse '/cmd arg1 arg2' and dispatch to the matching prompt."""
    if not user_input.startswith('/'):
        return None
    parts = user_input[1:].split()
    if not parts:
        return 'Empty slash command'
    cmd, positional = parts[0], parts[1:]

    async with MCPClient(command=command, args=args) as client:
        prompts = await client.list_prompts()
        target = next((p for p in prompts if p.name == cmd), None)
        if not target:
            return f'Unknown command: /{cmd}. Available: {[p.name for p in prompts]}'

        # Map positional args to declared argument names
        arg_dict = {}
        if target.arguments:
            for i, arg_meta in enumerate(target.arguments):
                if i < len(positional):
                    arg_dict[arg_meta.name] = positional[i]

        messages = await client.get_prompt(cmd, arg_dict)
        return {
            'prompt_name': cmd,
            'arguments': arg_dict,
            'messages': messages,
        }

# Test /format plan.md
result = await handle_slash('/format plan.md')
print('--- /format plan.md result ---')
print('Prompt:', result['prompt_name'])
print('Args:  ', result['arguments'])
print('Msgs:  ', len(result['messages']), 'messages')
print('First message preview:', str(result['messages'][0].content)[:150])
print()

# Test unknown command
result2 = await handle_slash('/notfound foo')
print('--- /notfound result ---')
print(result2)

## §8. 도구와 리소스와 프롬프트의 3자 협업 — 한 사이클의 전체 흐름

강의노트 §2.5 (라인 약 1356 부터 1385 까지)에는 세 컴포넌트가 협업하는 한 사이클의 흐름이 시퀀스 다이어그램으로 정리되어 있다. 이를 한국어로 풀어 단계별로 정리하면 다음과 같다.

먼저 사용자가 슬래시 명령으로 어떤 문서의 포맷팅을 요청하는 것에서 시작한다. 그러면 클라이언트가 슬래시 처리 함수로 입력을 파싱한다. 그다음 클라이언트가 프롬프트 목록 조회를 호출하여 서버의 프롬프트 카탈로그에서 포맷팅이라는 프롬프트를 찾는다. 카탈로그 조회가 끝나면 프롬프트 가져오기 메서드를 호출하여 보간된 메시지를 요청한다.

서버 측에서는 프롬프트 데코레이터로 등록된 문서 포맷팅 함수가 호출된다. 그 함수는 사용자 메시지 하나가 들어 있는 메시지 리스트를 반환하는데, 그 메시지 본문에는 문서 편집 도구를 사용하라는 지시가 담겨 있다. 이 메시지가 클라이언트로 돌아오면, 클라이언트는 그것을 그대로 클로드에게 전달한다. 그러면 클로드가 메시지의 지시에 따라 동작을 시작한다.

클로드가 처음 내놓는 응답은 도구 사용 요청이다. 즉 문서 편집 도구를 호출하라는 형태의 응답이 온다. 클라이언트는 그 요청을 받아 도구 호출 메서드로 서버에 도구 호출을 전달한다. 서버 측에서는 도구 데코레이터로 등록된 문서 편집 함수가 실행되어 문서 사전을 변경한다. 도구 실행이 끝나면 그 결과가 다시 클로드에게 전달되고, 마지막으로 클로드가 최종 답변 텍스트를 생성하여 반환한다.

이 흐름의 핵심은 무엇일까. 포맷팅 프롬프트의 본문 안에 문서 편집 도구를 사용하라는 지시가 명시적으로 들어 있다는 점이다. 그래서 클로드가 자연스럽게 그 도구를 호출하게 된다. 즉 프롬프트가 도구 사용을 유도하는 3자 연계의 출발점 역할을 하는 셈이다.

> [!finding] 강의노트 §2.5 의 결론 — 프롬프트는 다리다
> 강의노트는 다음과 같이 정리한다. 프롬프트는 사전 정의된 기능과 동적인 사용자 요구 사이의 다리 역할을 한다. 즉 복잡한 태스크의 구조화된 출발점을 제공하면서도, 매개변수화를 통해 유연성을 유지하는 절묘한 균형점이라는 의미다. 프롬프트가 너무 고정적이면 사용자의 다양한 요구를 받아낼 수 없고, 너무 자유로우면 일관성과 품질을 보장할 수 없다. 그 사이의 균형을 매개변수화로 잡는다는 것이 핵심이다.


In [ ]:
# Week_07.md §2.5 line ~1356 — 3자 협업 시뮬레이션 (Claude 호출은 mock)
# 실제 Claude 호출은 W04 (S3) Tool Use 노트북 참고

async def simulate_3way_collab(user_input: str):
    """Simulate the full /format -> prompt -> tool -> result chain."""
    print(f'>>> User: {user_input}')

    # Step 1: Parse slash command
    parsed = await handle_slash(user_input)
    if isinstance(parsed, str):
        print(parsed)
        return

    # Step 2: Show what would be sent to Claude
    print(f'\n[Step 1] Prompt fetched from MCP server:')
    print(f'  prompt_name = {parsed["prompt_name"]}')
    print(f'  args        = {parsed["arguments"]}')
    print(f'  messages[0] = {str(parsed["messages"][0].content)[:100]}...')

    # Step 3: Simulate Claude deciding to call edit_document
    print(f'\n[Step 2] (mock) Claude decides: tool_use("edit_document", ...)')

    # Step 4: Demonstrate that the tool exists on the server
    async with MCPClient(command=command, args=args) as client:
        tools = await client.list_tools()
        edit_tool = next((t for t in tools if t.name == 'edit_document'), None)
        print(f'\n[Step 3] Server provides the tool: {edit_tool.name if edit_tool else "NOT FOUND"}')
        if edit_tool:
            print(f'  description: {edit_tool.description}')

    print(f'\n[Step 4] Claude returns final markdown content as text response.')
    print(f'\n--- 3-way collab cycle complete ---')

await simulate_3way_collab('/format plan.md')

## §9. 다음 단계로

본 노트북에서 완성한 자산을 정리해 보자. 첫 번째 자산은 서버 파일의 최종 완성본이다. 도구 두 개와 리소스 두 개와 프롬프트 한 개를 모두 포함하여, 참조 구현인 도메인 폴더의 서버 파일과 동등한 수준에 도달했다. 두 번째 자산은 클라이언트 파일의 활성화다. 이미 정의되어 있던 프롬프트 목록 조회와 프롬프트 가져오기 두 메서드의 동작을 데모로 직접 검증해 보았다. 세 번째 자산은 슬래시 명령 파서다. 슬래시 처리 함수를 작성하여, 임의의 슬래시 명령을 적절한 프롬프트로 디스패치하는 로직을 구현하였다. 네 번째 자산은 3자 협업 시뮬레이션이다. 프롬프트가 도구 호출을 유도하는 패턴을 한 흐름으로 시연해 보았다.

**다음 노트북에서 다룰 내용**:

다음 노트북은 학생 자율 실습으로 구성된다. 새로운 도메인을 하나 선택하여 자신만의 엠시피 서버를 백지에서 구성하는 도전 과제다. 권장하는 도메인은 할 일 관리와 레시피 컬렉션과 개인 지식 베이스 세 가지 중 하나이지만, 학생이 직접 도메인을 정해도 좋다. 도전 과제의 슬롯은 정해져 있다. 도구 세 개와 리소스 두 개와 프롬프트 한 개를 채워서 자체적으로 동작하는 서버를 만들고, 클라이언트로 연결하여 모든 컴포넌트가 정상 작동하는지 자가 검증해야 한다.

**그 이후의 트랙**:

학생 자율 실습이 끝나면 도메인 응용 트랙으로 넘어간다. 그곳에서는 본 7 주차에서 익힌 패턴을 한국 건축공학 도메인에 적용한다. 한국 설계기준과 마이다스 해석 결과, 그리고 구조 검토 프롬프트 같은 실제 도메인 자산을 사용하여 산업적으로 의미 있는 엠시피 서버를 직접 만들어 보게 된다. 강의노트 §2.7 이 그 트랙의 텍스트 소스가 된다.

> [!ref] 강의노트 §2.6 (라인 약 1399-1472) → 다음 노트북
